# Demo: GraphCodeBERT Code Similarity\n\n[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jorge-martinez-gil/graphcodebert-interpretability/blob/main/examples/demo_similarity.ipynb)\n\nThis notebook demonstrates an end-to-end similarity workflow using GraphCodeBERT on Bubble Sort vs Insertion Sort.

## 1) Install and import dependencies\nIf you are running on Colab, uncomment the installation line.

In [ ]:
# !pip install torch transformers matplotlib numpy\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport torch\nfrom transformers import RobertaModel, RobertaTokenizer

## 2) Load GraphCodeBERT

In [ ]:
model_name = "microsoft/graphcodebert-base"\ntokenizer = RobertaTokenizer.from_pretrained(model_name)\nmodel = RobertaModel.from_pretrained(model_name)\nmodel.eval()

## 3) Define two code snippets

In [ ]:
bubble_sort = '''\ndef bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n - i - 1):\n            if arr[j] > arr[j + 1]:\n                arr[j], arr[j + 1] = arr[j + 1], arr[j]\n    return arr\n'''\n\ninsertion_sort = '''\ndef insertion_sort(arr):\n    for i in range(1, len(arr)):\n        key = arr[i]\n        j = i - 1\n        while j >= 0 and key < arr[j]:\n            arr[j + 1] = arr[j]\n            j -= 1\n        arr[j + 1] = key\n    return arr\n'''

## 4) Compute embedding-based cosine similarity

In [ ]:
def get_snippet_embedding(code: str) -> torch.Tensor:\n    inputs = tokenizer(code, return_tensors="pt", max_length=512, truncation=True, padding=True)\n    with torch.no_grad():\n        outputs = model(**inputs)\n    return outputs.last_hidden_state.mean(dim=1).squeeze(0)\n\nemb_bubble = get_snippet_embedding(bubble_sort)\nemb_insertion = get_snippet_embedding(insertion_sort)\n\nsimilarity = torch.nn.functional.cosine_similarity(emb_bubble.unsqueeze(0), emb_insertion.unsqueeze(0)).item()\nprint(f"Cosine similarity (Bubble Sort vs Insertion Sort): {similarity:.4f}")

## 5) Visualize the similarity score

In [ ]:
plt.figure(figsize=(5, 3))\nplt.bar(["Bubble vs Insertion"], [similarity], color="steelblue")\nplt.ylim(0, 1)\nplt.ylabel("Cosine Similarity")\nplt.title("GraphCodeBERT Similarity Demo")\nplt.tight_layout()\nplt.show()